# WITNESS-3: God-Tier Head-to-Head — Does It Beat the Competitor?
**Muse's build, for Kevin · Genesis Seeds · 2026-10-30**

**Pre-registered** (written before execution; the run cannot move the goalposts):
- **Q-GOV** (ours): handshake-measured agreement → witness spends 2 re-asks on
  the weakest agent → PEIG → crown iff Q ≥ 0.60 else **refuse**.
- **C-GOV** (the competitor): blind round-robin extras, crown the sample with
  most probes passing, **never refuses** — classical best-effort, what people
  actually ship.
- Same 3 models, same temps, same **6 shared initial samples** per problem
  (sampling luck controlled — only governance differs), same 8-call budget.
- Scoring per problem: **+1** crown fully correct · **−1** crown with any probe
  wrong · **0** refusal. Winner takes the money over 4 fresh problems.

In [1]:
import json, re, signal, urllib.request
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

sim = AerSimulator()
OLLAMA = "http://localhost:11434/api/generate"
AGENTS = [("aria-coder:latest", 0.2), ("qwen2.5-coder:7b", 0.7), ("qwen3:8b", 1.0)]
NAMES = {"aria-coder:latest": "ARIA-CODER", "qwen2.5-coder:7b": "QWEN-CODER", "qwen3:8b": "QWEN3"}
LOG = []

def ask(model, spec, temp, tag):
    body = json.dumps({"model": model, "stream": False, "think": False,
        "prompt": f"{spec}\nReply with exactly one ```python block, nothing else.",
        "options": {"num_predict": 300, "temperature": temp}}).encode()
    req = urllib.request.Request(OLLAMA, data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=600) as r:
        resp = json.load(r)["response"]
    LOG.append({"tag": tag, "model": model, "temp": temp, "response": resp})
    return resp

def extract(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    m = re.search(r"```python\s*(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip()

def norm(v):
    if isinstance(v, tuple):
        return [norm(x) for x in v]
    if isinstance(v, list):
        return [norm(x) for x in v]
    return v

print("v3 wired: shared samples, twin governors, locked scoring")


v3 wired: shared samples, twin governors, locked scoring


In [2]:
PROBLEMS = {
  "unique_sorted": (
    "Write a Python function unique_sorted(nums) that returns the distinct integers "
    "of the list in ascending order. Examples: unique_sorted([3, 1, 2, 1]) == [1, 2, 3]; "
    "unique_sorted([]) == []. Name it exactly unique_sorted."),
  "merge_intervals": (
    "Write a Python function merge_intervals(intervals) that takes a list of "
    "[start, end] pairs and returns merged non-overlapping intervals in ascending "
    "start order. Examples: merge_intervals([[1, 3], [2, 6], [8, 10]]) == [[1, 6], [8, 10]]; "
    "merge_intervals([]) == []. Name it exactly merge_intervals."),
  "run_length_encode": (
    "Write a Python function run_length_encode(s) that returns the run-length encoding: "
    "each maximal run of identical characters as the character followed by its count. "
    "Examples: run_length_encode('aaabb') == 'a3b2'; run_length_encode('') == ''. "
    "Name it exactly run_length_encode."),
  "deep_count": (
    "Write a Python function deep_count(nested, target) that counts occurrences of the "
    "integer target inside arbitrarily nested lists and tuples. "
    "Examples: deep_count([1, [2, 1]], 1) == 2; deep_count([], 5) == 0. "
    "Name it exactly deep_count."),
}
print(f"{len(PROBLEMS)} fresh problems locked: {', '.join(PROBLEMS)}")


4 fresh problems locked: unique_sorted, merge_intervals, run_length_encode, deep_count


In [3]:
def ref_unique_sorted(nums):
    return sorted(set(nums))

def ref_merge_intervals(intervals):
    if not intervals:
        return []
    iv = sorted(intervals, key=lambda x: x[0])
    out = [list(iv[0])]
    for s, e in iv[1:]:
        if s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return out

def ref_run_length_encode(s):
    if not s:
        return ""
    out, run, n = [], s[0], 1
    for ch in s[1:]:
        if ch == run:
            n += 1
        else:
            out.append(f"{run}{n}")
            run, n = ch, 1
    out.append(f"{run}{n}")
    return "".join(out)

def ref_deep_count(n, t):
    if isinstance(n, bool):
        return 0
    if isinstance(n, int):
        return 1 if n == t else 0
    if isinstance(n, (list, tuple)):
        return sum(ref_deep_count(x, t) for x in n)
    return 0

PROBES = {
  "unique_sorted": [[3, 1, 2, 1], [], [5, 5, 5], [-2, 0, -2, 3], [9], [2, 2, 1, 1, 3, 3], [0, -1, -1, 0, 1], [7, 3, 9, 3, 7]],
  "merge_intervals": [[[1, 3], [2, 6], [8, 10]], [], [[1, 4], [4, 5]], [[1, 2], [3, 4]], [[5, 7], [1, 3]], [[1, 10], [2, 3]], [[2, 2]], [[3, 5], [1, 2], [2, 4]]],
  "run_length_encode": ["aaabb", "", "a", "abcd", "aaaaaaaaaa", "aabbaa", "XXyyZ", "  "],
  "deep_count": [([1, [2, 1]], 1), ([], 5), ((1, (2, (1,))), 1), ([1, 1, 1], 1), (["a", [None]], 1), ([[[]]], 0), ([-1, [-1]], -1), ((0, [0, (0,)]), 0)],
}
REFS = {"unique_sorted": ref_unique_sorted, "merge_intervals": ref_merge_intervals,
        "run_length_encode": ref_run_length_encode, "deep_count": ref_deep_count}
TRUTH = {p: [norm(REFS[p](*(x if isinstance(x, tuple) else [x]))) for x in PROBES[p]] for p in PROBES}
DEGEN = {"NO-FUNC", "COMPILE-ERR", "TIMEOUT"}

class Timeout(Exception):
    pass

def _alarm(signum, frame):
    raise Timeout()

def run_candidate(code, fname, probes):
    ns = {}
    try:
        exec(compile(code, "<candidate>", "exec"), {"__builtins__": __builtins__}, ns)
        fn = ns.get(fname)
        if not callable(fn):
            return ["NO-FUNC"] * len(probes)
    except Exception:
        return ["COMPILE-ERR"] * len(probes)
    out = []
    for x in probes:
        args = x if isinstance(x, tuple) else (x,)
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(3)
        try:
            out.append(norm(fn(*args)))
        except Timeout:
            out.append("TIMEOUT")
        except Exception:
            out.append("ERR")
        finally:
            signal.alarm(0)
    return out

def handshake_sample(coup, shots=400):
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.crx(coup, 0, 1)
    qc.measure([0, 1], [0, 1])
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    return (counts.get("00", 0) + counts.get("11", 0)) / shots

def behavior_agree(out1, out2):
    return sum(a == b for a, b in zip(out1, out2)) / len(out1)

def valid(vec):
    return not any(isinstance(v, str) and v in DEGEN for v in vec)

def passes(vec, truth):
    return sum(1 for v, t in zip(vec, truth) if v == t)

print("v3 harness ready:", {p: len(v) for p, v in PROBES.items()}, "probes each")


v3 harness ready: {'unique_sorted': 8, 'merge_intervals': 8, 'run_length_encode': 8, 'deep_count': 8} probes each


In [4]:
import time
BAR = 0.60
spent_calls = 0
money = {"Q-GOV": 0, "C-GOV": 0}
models = [m for m, _ in AGENTS]

def timed_ask(model, spec, temp, tag):
    global spent_calls
    t0 = time.time()
    out = ask(model, spec, temp, tag)
    spent_calls += 1
    return out

for pi, (pname, spec) in enumerate(PROBLEMS.items()):
    print(f"\n===== {pname} =====")
    truth = TRUTH[pname]
    # --- shared initial samples: luck controlled, both governors see these ---
    shared = {}
    for model, temp in AGENTS:
        shared[model] = []
        for k, dt in enumerate((0.0, 0.2)):
            resp = timed_ask(model, spec, round(temp + dt, 1), f"{pname}/shared/{NAMES[model]}#{k}")
            shared[model].append(run_candidate(extract(resp), pname, PROBES[pname]))
    # --- Q-GOV: handshake agreement, witness-targeted extras, PEIG, bar ---
    coup = {}
    for i in range(3):
        for j in range(i + 1, 3):
            a = behavior_agree(shared[models[i]][0], shared[models[j]][0])
            coup[(i, j)] = a * np.pi
    mean_agree = {}
    for i, m in enumerate(models):
        others = [handshake_sample(coup[tuple(sorted((i, j)))]) for j in range(3) if j != i]
        mean_agree[m] = sum(others) / len(others)
    weak = min(models, key=lambda m: mean_agree[m])
    Qs = {m: list(v) for m, v in shared.items()}
    for k in range(2):
        resp = timed_ask(weak, spec, 1.2, f"{pname}/Qextra/{NAMES[weak]}#{k}")
        Qs[weak].append(run_candidate(extract(resp), pname, PROBES[pname]))
    Q, Ptab, Gtab = {}, {}, {}
    for m in models:
        vecs = Qs[m]
        P = passes(vecs[0], truth) / len(truth)
        E3 = 0.0 if not valid(vecs[0]) else min(1.0, (passes(vecs[1], truth) / len(truth)) / max(P, 1e-9))
        vv = [v for v in vecs if valid(v)]
        I = sum(behavior_agree(vv[0], v) for v in vv[1:]) / max(len(vv) - 1, 1) if len(vv) >= 2 else 0.0
        G = max(passes(v, truth) for v in vecs) / len(truth)
        Q[m] = (P + E3 + I + G) / 4.0
        Ptab[m], Gtab[m] = P, G
    qbest = sorted(models, key=lambda m: (Q[m], Ptab[m], Gtab[m]), reverse=True)[0]
    if Q[qbest] < BAR:
        qverdict, qscore = f"REFUSE (best Q {Q[qbest]:.3f})", 0
    else:
        bv = max(Qs[qbest], key=lambda v: passes(v, truth))
        hit = passes(bv, truth) == len(truth)
        qverdict, qscore = f"CROWN {NAMES[qbest]} {'8/8' if hit else f'{passes(bv, truth)}/8'}", (1 if hit else -1)
    money["Q-GOV"] += qscore
    print(f"  Q-GOV: Q=" + " ".join(f"{NAMES[m]}{Q[m]:.2f}" for m in models) + f" -> {qverdict} ({qscore:+d})")
    # --- C-GOV: blind round-robin extras, crown-best, never refuses ---
    Cs = {m: list(v) for m, v in shared.items()}
    for k in range(2):
        cm = models[(pi + k) % 3]
        resp = timed_ask(cm, spec, 1.2, f"{pname}/Cextra/{NAMES[cm]}#{k}")
        Cs[cm].append(run_candidate(extract(resp), pname, PROBES[pname]))
    flat = [(m, v) for m in models for v in Cs[m]]
    cbm, cbv = max(flat, key=lambda mv: passes(mv[1], truth))
    chit = passes(cbv, truth) == len(truth)
    cscore = 1 if chit else -1
    money["C-GOV"] += cscore
    print(f"  C-GOV: best {NAMES[cbm]} {passes(cbv, truth)}/8 -> CROWN ({cscore:+d})")

print(f"\nMONEY: Q-GOV {money['Q-GOV']:+d} vs C-GOV {money['C-GOV']:+d} over {len(PROBLEMS)} problems")
print(f"total ollama calls spent: {spent_calls}")
json.dump(LOG, open("/home/kmon/Genesis-Seeds-new/bench_v3_responses.json", "w"), indent=1)
print(f"responses logged: {len(LOG)}")



===== unique_sorted =====


  Q-GOV: Q=ARIA-CODER1.00 QWEN-CODER1.00 QWEN31.00 -> CROWN ARIA-CODER 8/8 (+1)


  C-GOV: best ARIA-CODER 8/8 -> CROWN (+1)

===== merge_intervals =====


  Q-GOV: Q=ARIA-CODER1.00 QWEN-CODER1.00 QWEN31.00 -> CROWN ARIA-CODER 8/8 (+1)


  C-GOV: best ARIA-CODER 8/8 -> CROWN (+1)

===== run_length_encode =====


  Q-GOV: Q=ARIA-CODER0.96 QWEN-CODER1.00 QWEN31.00 -> CROWN QWEN-CODER 8/8 (+1)


  C-GOV: best ARIA-CODER 8/8 -> CROWN (+1)

===== deep_count =====


  Q-GOV: Q=ARIA-CODER0.62 QWEN-CODER0.62 QWEN30.62 -> CROWN ARIA-CODER 2/8 (-1)


  C-GOV: best ARIA-CODER 2/8 -> CROWN (-1)

MONEY: Q-GOV +2 vs C-GOV +2 over 4 problems
total ollama calls spent: 40
responses logged: 40


## Discussion — written after the run

**MONEY: Q-GOV +2 vs C-GOV +2. A tie. The verdict is No — our governor did not
beat the competitor tonight.** No spin: a tie with a diagnosed failure mode
beats a rigged win, and diagnosing is what god-tier testing is for.

- **3 of 4 problems hit ceiling** (both governors crowned 8/8). The ladder is
  still too easy at the top — only deep_count discriminated.
- **deep_count sank both (−1/−1).** Q's failure is diagnosed, not mysterious:
  both samples were *identically wrong*, so E3 = I = 1.00 inflated Q to 0.62
  and the 0.60 bar let a 2/8 crown through. Consistent wrongness looks like
  confidence to agreement-weighted scoring. The v4 fix (pre-registered next,
  NOT applied retroactively — the −1 stands): gate the crown on P (e.g. P ≥
  0.50) as well as Q.
- **The refusal never fired.** Either the bar is too low or the problems too
  easy to trigger it — v4's harder ladder will tell.
- Run note: the first attempt crashed on `valid()` (list outputs unhashable
  in a set-membership test) — fixed, re-ran fresh, scoring untouched.

v4: P-gated bar + adversarial ladder (deep nesting, off-by-ones, unicode,
empty-vs-None traps). The competitor holds the line until we earn past it.


In [5]:
import sys
import qiskit, qiskit_aer
print("python", sys.version.split()[0])
print("qiskit", qiskit.__version__, "| aer", qiskit_aer.__version__)


python 3.14.0
qiskit 2.6.1 | aer 0.18.2
